In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
roboflow_api_key = os.getenv('roboflow_api_key')

In [ ]:
# NOTE: uncommend this if to download dataset
from roboflow import Roboflow
rf = Roboflow(api_key=roboflow_api_key)

project = rf.workspace("helmet-and-number-plate-detection-project").project("helmet-and-number-plate-detection-for-motorbike-safety-iityz")
version = project.version(3)
dataset = version.download("yolov11")

In [ ]:
# Visualize dataset
import cv2
import matplotlib.pyplot as plt
import os

# Path to your images and annotations
image_dir = '/kaggle/working/helmet-1/train/images'
label_dir = '/kaggle/working/helmet-1/train/labels'

# Get a list of image files
image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

# Visualize a few images with bounding boxes
for image_file in image_files[:5]:  # Display only the first 5 images for brevity
    # Read image
    image_path = os.path.join(image_dir, image_file)
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    error = []
    
    # Read corresponding annotation file
    annotation_path = os.path.join(label_dir, image_file.replace('.jpg', '.txt'))
    if os.path.exists(annotation_path):
        with open(annotation_path, 'r') as file:
            for line in file.readlines():
                parts = line.strip().split()
                if len(parts) <= 5:
                    class_id, x_center, y_center, width, height = map(float, line.strip().split())
                    
                    # Convert normalized coordinates to pixel coordinates
                    img_height, img_width = img.shape[:2]
                    x_center *= img_width
                    y_center *= img_height
                    width *= img_width
                    height *= img_height
                    
                    # Calculate top-left corner
                    top_left_x = int(x_center - width / 2)
                    top_left_y = int(y_center - height / 2)
                    bottom_right_x = int(x_center + width / 2)
                    bottom_right_y = int(y_center + height / 2)
                    
                    # Draw bounding box
                    cv2.rectangle(img, (top_left_x, top_left_y), (bottom_right_x, bottom_right_y), (255, 0, 0), 2)
                    cv2.putText(img, f'Class {int(class_id)}', (top_left_x, top_left_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
                else:
                    print(image_path)
                    print(annotation_path)
    # Display the image
    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis('off')
    
    plt.show()